In [289]:
import os
import re
import html
import requests
import feedparser
import pandas as pd
import numpy as np

from datetime import datetime
from dotenv import load_dotenv
import mysql.connector

In [290]:
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")


In [291]:
nse_feeds = {
    "online_announcement": "https://nsearchives.nseindia.com/content/RSS/Online_announcements.xml",
    "financial_result": "https://nsearchives.nseindia.com/content/RSS/Financial_Results.xml",
    "board_meeting": "https://nsearchives.nseindia.com/content/RSS/Board_Meetings.xml",
    "corporate_action": "https://nsearchives.nseindia.com/content/RSS/Corporate_action.xml",
    "annual_report": "https://nsearchives.nseindia.com/content/RSS/Annual_Reports.xml",
    "insider_trading": "https://nsearchives.nseindia.com/content/RSS/InsiderTrading.xml",
    "shareholding_pattern": "https://nsearchives.nseindia.com/content/RSS/Shareholding_Pattern.xml",
    "corporate_governance": "https://nsearchives.nseindia.com/content/RSS/Corporate_Governance.xml",
    "related_party_transaction": "https://nsearchives.nseindia.com/content/RSS/Related_Party_Trans.xml"
}

print("Total feeds:", len(nse_feeds))

Total feeds: 9


In [292]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    nse_feeds["online_announcement"],
    headers=headers,
    timeout=20
)

print("Status:", response.status_code)
print("Content length:", len(response.content))

Status: 200
Content length: 79823


In [293]:
# Clean text

def clean_text(value):
    if value is None:
        return None
    
    value = html.unescape(str(value))
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"\s+", " ", value)
    
    return value.strip()

In [294]:
def parse_date(value):
    if not value:
        return pd.NaT
    
    try:
        return pd.to_datetime(value, errors="coerce")
    except:
        return pd.NaT

In [295]:
def fetch_nse_feed(feed_type, feed_url):
    
    try:
        response = requests.get(
            feed_url,
            headers=headers,
            timeout=30
        )
        
        response.raise_for_status()
        
        feed = feedparser.parse(response.content)
        
        records = []
        
        for entry in feed.entries:
            
            company_name = (
                entry.get("companyname")
                or entry.get("company_name")
                or entry.get("title")
            )
            
            headline = (
                entry.get("description")
                or entry.get("summary")
                or entry.get("title")
            )
            
            published_at = (
                entry.get("published")
                or entry.get("pubdate")
                or entry.get("updated")
            )
            
            event_date = (
                entry.get("eventdate")
                or entry.get("event_date")
            )
            
            url = (
                entry.get("link")
                or entry.get("url")
            )
            
            records.append({
                "company_name": clean_text(company_name),
                "headline": clean_text(headline),
                "feed_type": feed_type,
                "published_at": parse_date(published_at),
                "event_date": parse_date(event_date),
                "url": url,
                "source": "NSE"
            })
        
        print(f"{feed_type}: {len(records)} records")
        
        return records
    
    except Exception as e:
        print(f"{feed_type}: ERROR -> {e}")
        return []

In [296]:
# Fetch all NSE Feeds

all_records = []

for feed_type, feed_url in nse_feeds.items():

    records = fetch_nse_feed(
        feed_type,
        feed_url
    )

    all_records.extend(records)

print("\nTotal raw records:", len(all_records))

online_announcement: 207 records
financial_result: 4 records
board_meeting: 0 records
corporate_action: 86 records
annual_report: 20 records
insider_trading: 0 records
shareholding_pattern: 0 records
corporate_governance: 0 records
related_party_transaction: 20 records

Total raw records: 337


In [297]:
nse_news = pd.DataFrame(all_records)

print(nse_news.shape)
nse_news.head()

(337, 7)


,company_name,headline,feed_type,published_at,event_date,url,source
0,Elgi Equipments Limited,Elgi Equipments Limited has informed the Exchange about General Updates |SUBJECT: General Updates,online_announcement,2026-08-29 07:48:32,NaT,https://nsearchives.nseindia.com/corporate/ELGIEQUIP_29082026074809_Intimation_-_US_Tariff_Refund.pdf,NSE
1,HDFC Gold ETF,HDFCGOLD : HDFC Asset Management Company Limited has informed the Exchange that the Net Asset Value (per unit) of HDFC Mutual Fund-HDFC Gold Excha...,online_announcement,2026-08-29 07:01:00,NaT,None,NSE
2,SBI Mutual Fund - SBI Nifty200 Value 30 ETF,SBIVALETF : SBI Funds Management Limited has informed the Exchange that the Net Asset Value (per unit) of SBI Mutual Fund - SBI Nifty200 Value 30 ...,online_announcement,2026-08-29 07:01:00,NaT,None,NSE
3,quant Mutual Fund - qsif Active Asset Allocator Long-Short Fund RR,QSIFAARR : quant Money Managers Limited has informed the Exchange that the Net Asset Value (per unit) of quant Mutual Fund - qsif Active Asset All...,online_announcement,2026-08-29 07:01:00,NaT,None,NSE
4,quant Mutual Fund - qsif Active Asset Allocator Long-Short Fund RG,QSIFAARG : quant Money Managers Limited has informed the Exchange that the Net Asset Value (per unit) of quant Mutual Fund - qsif Active Asset All...,online_announcement,2026-08-29 07:01:00,NaT,None,NSE


In [298]:
# Basic Cleaning

nse_news = nse_news[
    [
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "event_date",
        "url",
        "source"
    ]
].copy()

nse_news["company_name"] = nse_news["company_name"].astype(str).str.strip()
nse_news["headline"] = nse_news["headline"].astype(str).str.strip()
nse_news["url"] = nse_news["url"].astype(str).str.strip()

nse_news["published_at"] = pd.to_datetime(
    nse_news["published_at"],
    errors="coerce"
)

nse_news["event_date"] = pd.to_datetime(
    nse_news["event_date"],
    errors="coerce"
)

print("Total records:", len(nse_news))

Total records: 337


In [299]:
def get_date_status(row):
    
    published = pd.notna(row["published_at"])
    event = pd.notna(row["event_date"])
    
    if published and event:
        return "published_and_event"
    
    if published:
        return "published_only"
    
    if event:
        return "event_only"
    
    return "missing"


In [300]:
nse_news["date_status"] = nse_news.apply(
    get_date_status,
    axis=1
)

nse_news["date_status"].value_counts()

date_status
published_only    317
missing            20
Name: count, dtype: int64

In [301]:
print("========== RAW NSE VALIDATION ==========")

print("Total:", len(nse_news))

print("\nFeed distribution:")
print(nse_news["feed_type"].value_counts())

print("\nMissing values:")
print(nse_news.isna().sum())

print("\nDuplicate URLs:")
print(nse_news["url"].duplicated().sum())

print("\nDuplicate headlines:")
print(nse_news["headline"].duplicated().sum())

========== RAW NSE VALIDATION ==========
Total: 337

Feed distribution:
feed_type
online_announcement          207
corporate_action              86
annual_report                 20
related_party_transaction     20
financial_result               4
Name: count, dtype: int64

Missing values:
company_name      0
headline          0
feed_type         0
published_at     20
event_date      337
url               0
source            0
date_status       0
dtype: int64

Duplicate URLs:
265

Duplicate headlines:
47


In [323]:
# Connect mySQL

db = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

print("MySQL connection:", db.is_connected())

MySQL connection: True


In [303]:
query = """
SELECT
    id AS company_id,
    name AS db_company_name,
    symbol,
    isin
FROM companies
"""

companies_df = pd.read_sql(
    query,
    db
)

print("Companies:", len(companies_df))
companies_df.head()

Companies: 3117


,company_id,db_company_name,symbol,isin
0,5,20 Microns Limited,20MICRONS,INE144J01027
1,6,21st Century Management Services Limited,21STCENMGM,INE253B01015
2,7,360 ONE WAM LIMITED,360ONE,INE466L01038
3,8,3B Blackbio Dx Limited,3BBLACKBIO,INE994E01018
4,9,3i Infotech Limited,3IINFOLTD,INE748C01038


In [304]:
def normalize_company_name(name):
    
    if pd.isna(name):
        return ""
    
    name = str(name).upper()
    
    # Remove HTML
    name = re.sub(r"<[^>]+>", " ", name)
    
    # Replace common punctuation
    name = name.replace("&", " AND ")
    
    # Remove bracket content
    name = re.sub(r"\([^)]*\)", " ", name)
    
    # Remove common company suffixes
    suffixes = [
        "LIMITED",
        "LTD",
        "LTD.",
        "PVT",
        "PRIVATE",
        "PUBLIC",
        "INC",
        "INCORPORATED"
    ]
    
    for suffix in suffixes:
        name = re.sub(
            rf"\b{re.escape(suffix)}\b",
            " ",
            name
        )
    
    # Keep alphanumeric
    name = re.sub(r"[^A-Z0-9 ]", " ", name)
    
    # Remove extra spaces
    name = re.sub(r"\s+", " ", name)
    
    return name.strip()

In [305]:
nse_news["clean_company_name"] = (
    nse_news["company_name"]
    .apply(normalize_company_name)
)

companies_df["clean_company_name"] = (
    companies_df["db_company_name"]
    .apply(normalize_company_name)
)

print(nse_news[
    ["company_name", "clean_company_name"]
].head(20))

                                                            company_name  \
0                                                Elgi Equipments Limited   
1                                                          HDFC Gold ETF   
2                            SBI Mutual Fund - SBI Nifty200 Value 30 ETF   
3     quant Mutual Fund - qsif Active Asset Allocator Long-Short Fund RR   
4     quant Mutual Fund - qsif Active Asset Allocator Long-Short Fund RG   
5     quant Mutual Fund - qsif Active Asset Allocator Long-Short Fund DP   
6     quant Mutual Fund - qsif Active Asset Allocator Long-Short Fund DG   
7                           360 ONE Mutual Fund - 360 ONE MSCI India ETF   
8                       Groww Mutual Fund - Groww Nifty Private Bank ETF   
9                     Invesco Mutual Fund - Invesco India BSE Sensex ETF   
10        quant Mutual Fund - qsif Hybrid Long-Short Fund - RP - IDCW PO   
11                           Groww Mutual Fund - Groww Nifty Cements ETF   
12          

In [306]:
company_mapping = (
    companies_df
    [
        [
            "company_id",
            "db_company_name",
            "symbol",
            "isin",
            "clean_company_name"
        ]
    ]
    .drop_duplicates("clean_company_name")
)

nse_news = nse_news.merge(
    company_mapping,
    on="clean_company_name",
    how="left"
)

print("Total NSE records:", len(nse_news))

print(
    "Matched records:",
    nse_news["company_id"].notna().sum()
)

print(
    "Unmatched records:",
    nse_news["company_id"].isna().sum()
)

Total NSE records: 337
Matched records: 66
Unmatched records: 271


In [307]:
unmatched = nse_news[
    nse_news["company_id"].isna()
].copy()

print(
    "Unmatched unique companies:",
    unmatched["company_name"].nunique()
)

unmatched[
    [
        "company_name",
        "clean_company_name",
        "feed_type"
    ]
].drop_duplicates().sort_values(
    "company_name"
).head(100)

Unmatched unique companies: 268


,company_name,clean_company_name,feed_type
30,360 ONE Mutual Fund - 360 ONE Gold ETF,360 ONE MUTUAL FUND 360 ONE GOLD ETF,online_announcement
7,360 ONE Mutual Fund - 360 ONE MSCI India ETF,360 ONE MUTUAL FUND 360 ONE MSCI INDIA ETF,online_announcement
129,360 ONE Mutual Fund - 360 ONE Silver ETF,360 ONE MUTUAL FUND 360 ONE SILVER ETF,online_announcement
238,AGI Greenpac Limited - Ex-Date: 15-Sep-2026,AGI GREENPAC EX DATE 15 SEP 2026,corporate_action
239,AIA Engineering Limited - Ex-Date: 04-Sep-2026,AIA ENGINEERING EX DATE 04 SEP 2026,corporate_action
...,...,...,...
12,Groww Mutual Fund - Groww Nifty PSU Bank ETF,GROWW MUTUAL FUND GROWW NIFTY PSU BANK ETF,online_announcement
8,Groww Mutual Fund - Groww Nifty Private Bank ETF,GROWW MUTUAL FUND GROWW NIFTY BANK ETF,online_announcement
61,Groww Mutual Fund - Groww Nifty Realty ETF,GROWW MUTUAL FUND GROWW NIFTY REALTY ETF,online_announcement
32,Groww Mutual Fund - Groww Nifty Smallcap 250 ETF,GROWW MUTUAL FUND GROWW NIFTY SMALLCAP 250 ETF,online_announcement


In [308]:
matched_news = nse_news[
    nse_news["company_id"].notna()
].copy()

print("Matched records:", len(matched_news))

Matched records: 66


In [309]:
matched_news["company_id"] = (
    matched_news["company_id"]
    .astype(int)
)

matched_news["symbol"] = (
    matched_news["symbol"]
    .astype(str)
)

matched_news["isin"] = (
    matched_news["isin"]
    .astype(str)
)

In [310]:
final_news = matched_news[
    [
        "company_id",
        "symbol",
        "isin",
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "event_date",
        "date_status",
        "url",
        "source"
    ]
].copy()

print(final_news.shape)
final_news.head()

(66, 11)


,company_id,symbol,isin,company_name,headline,feed_type,published_at,event_date,date_status,url,source
0,666,ELGIEQUIP,INE285A01027,Elgi Equipments Limited,Elgi Equipments Limited has informed the Exchange about General Updates |SUBJECT: General Updates,online_announcement,2026-08-29 07:48:32,NaT,published_only,https://nsearchives.nseindia.com/corporate/ELGIEQUIP_29082026074809_Intimation_-_US_Tariff_Refund.pdf,NSE
180,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,SIGNPOST INDIA LIMITED has informed the Exchange about Notice of Shareholders Meeting for Annual General Meeting to be held on 23-Sep-2026 |SUBJEC...,online_announcement,2026-08-29 06:32:55,NaT,published_only,https://nsearchives.nseindia.com/corporate/xbrl/NOTICE_OF_SHAREHOLDERS_MEETINGS_22519_WebXMLFile_20260829_063254130.xml,NSE
181,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,Signpost India Limited has informed the Exchange regarding 'Regulation 30 of the Securities and Exchange Board of India (Listing Obligations and D...,online_announcement,2026-08-29 05:29:08,NaT,published_only,https://nsearchives.nseindia.com/corporate/SIGNPOST_29082026051802_SIL_Weblink_intimation.pdf,NSE
182,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,"Signpost India Limited has informed the Exchange regarding Notice of Annual General Meeting to be held on September 23, 2026 |SUBJECT: Shareholder...",online_announcement,2026-08-29 05:25:44,NaT,published_only,https://nsearchives.nseindia.com/corporate/SIGNPOST_29082026045744_Sil__AGM_notice_29082026.pdf,NSE
183,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,Signpost India Limited has informed the Exchange regarding 'Disclosure under Regulation 30 of SEBI (Listing Obligations and Disclosure Requirement...,online_announcement,2026-08-29 05:23:00,NaT,published_only,https://nsearchives.nseindia.com/corporate/SIGNPOST_29082026044335_SIL_SE_Reg_30_TDS_Communication_29082026.pdf,NSE


In [311]:
before = len(final_news)

final_news = final_news.drop_duplicates(
    subset=[
        "company_id",
        "feed_type",
        "headline",
        "url"
    ]
).copy()

after = len(final_news)

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

Before: 66
After: 66
Removed: 0


In [312]:
duplicate_urls = (
    final_news[
        final_news["url"].duplicated(keep=False)
    ]
    .sort_values("url")
)

print("Duplicate URL records:", len(duplicate_urls))

duplicate_urls.head(20)

Duplicate URL records: 4


,company_id,symbol,isin,company_name,headline,feed_type,published_at,event_date,date_status,url,source
209,95,AHLEAST,INE926K01017,Asian Hotels (West) Limited,RELATING TO:Annual |AUDITED/UNAUDITED:Audited |CUMULATIVE/NON-CUMULATIVE:Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |IND AS/ NON I...,financial_result,2026-08-24 13:25:32,NaT,published_only,https://archives.nseindia.com/corporate/xbrl/INDAS_121281_1717227_24082026012530.xml,NSE
210,95,AHLEAST,INE926K01017,Asian Hotels (West) Limited,RELATING TO:Fourth Quarter |AUDITED/UNAUDITED:Audited |CUMULATIVE/NON-CUMULATIVE:Non-Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |I...,financial_result,2026-08-24 13:25:31,NaT,published_only,https://archives.nseindia.com/corporate/xbrl/INDAS_121281_1717227_24082026012530.xml,NSE
207,95,AHLEAST,INE926K01017,Asian Hotels (West) Limited,RELATING TO:Annual |AUDITED/UNAUDITED:Audited |CUMULATIVE/NON-CUMULATIVE:Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Consolidated |IND AS/ NON IND A...,financial_result,2026-08-24 17:39:16,NaT,published_only,https://archives.nseindia.com/corporate/xbrl/INDAS_121282_1717387_24082026053915.xml,NSE
208,95,AHLEAST,INE926K01017,Asian Hotels (West) Limited,RELATING TO:Fourth Quarter |AUDITED/UNAUDITED:Audited |CUMULATIVE/NON-CUMULATIVE:Non-Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Consolidated |IND A...,financial_result,2026-08-24 17:39:15,NaT,published_only,https://archives.nseindia.com/corporate/xbrl/INDAS_121282_1717387_24082026053915.xml,NSE


In [313]:
print("========== FINAL VALIDATION ==========")

print("Total records:", len(final_news))

print("\nFeed distribution:")
print(
    final_news["feed_type"]
    .value_counts()
)

print("\nMissing values:")
print(
    final_news[
        [
            "company_id",
            "symbol",
            "isin",
            "company_name",
            "headline",
            "feed_type",
            "published_at",
            "event_date",
            "date_status",
            "url",
            "source"
        ]
    ].isna().sum()
)

print("\nDuplicate records:")
print(
    final_news.duplicated(
        subset=[
            "company_id",
            "feed_type",
            "headline",
            "url"
        ]
    ).sum()
)

print("\nPublished date:")

print(
    final_news["published_at"].min()
)

print(
    final_news["published_at"].max()
)

print("\nEvent date:")

print(
    final_news["event_date"].min()
)

print(
    final_news["event_date"].max()
)

========== FINAL VALIDATION ==========
Total records: 66

Feed distribution:
feed_type
online_announcement          28
annual_report                20
related_party_transaction    14
financial_result              4
Name: count, dtype: int64

Missing values:
company_id       0
symbol           0
isin             0
company_name     0
headline         0
feed_type        0
published_at    20
event_date      66
date_status      0
url              0
source           0
dtype: int64

Duplicate records:
0

Published date:
2025-02-21 15:12:33
2026-08-29 07:48:32

Event date:
NaT
NaT


In [314]:
required_columns = [
    "company_id",
    "symbol",
    "headline",
    "feed_type",
    "url"
]

print("========== REQUIRED FIELD CHECK ==========")

for column in required_columns:
    
    missing = final_news[column].isna().sum()
    
    print(
        f"Missing {column}: {missing}"
    )

========== REQUIRED FIELD CHECK ==========
Missing company_id: 0
Missing symbol: 0
Missing headline: 0
Missing feed_type: 0
Missing url: 0


In [315]:
cursor = db.cursor()

cursor.execute("""
    DESCRIBE news
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('id', 'bigint', 'NO', 'PRI', None, 'auto_increment')
('company_id', 'int', 'NO', 'MUL', None, '')
('symbol', 'varchar(50)', 'NO', '', None, '')
('isin', 'varchar(20)', 'NO', '', None, '')
('company_name', 'varchar(255)', 'NO', '', None, '')
('headline', 'text', 'NO', '', None, '')
('feed_type', 'varchar(100)', 'NO', '', None, '')
('published_at', 'datetime', 'YES', '', None, '')
('event_date', 'date', 'YES', '', None, '')
('date_status', 'varchar(30)', 'NO', '', None, '')
('url', 'text', 'NO', '', None, '')
('source', 'varchar(50)', 'NO', '', None, '')
('created_at', 'timestamp', 'YES', '', 'CURRENT_TIMESTAMP', 'DEFAULT_GENERATED')


True

In [316]:
cursor = db.cursor()

cursor.execute("SHOW TABLES")

tables = cursor.fetchall()

for table in tables:
    print(table[0])

cursor.close()

companies
industries
news
sectors
stock_prices


True

In [317]:
cursor = db.cursor()

create_news_table = """
CREATE TABLE IF NOT EXISTS news (
    id BIGINT AUTO_INCREMENT PRIMARY KEY,

    company_id INT NOT NULL,
    symbol VARCHAR(50) NOT NULL,
    isin VARCHAR(20) NOT NULL,
    company_name VARCHAR(255) NOT NULL,

    headline TEXT NOT NULL,
    feed_type VARCHAR(100) NOT NULL,

    published_at DATETIME NULL,
    event_date DATE NULL,
    date_status VARCHAR(30) NOT NULL,

    url TEXT NOT NULL,
    source VARCHAR(50) NOT NULL,

    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT fk_news_company
        FOREIGN KEY (company_id)
        REFERENCES companies(id)
        ON DELETE CASCADE
        ON UPDATE CASCADE
);
"""

cursor.execute(create_news_table)
db.commit()

cursor.close()

print("news table created successfully.")

news table created successfully.


In [318]:
cursor = db.cursor()

cursor.execute("DESCRIBE news")

for row in cursor.fetchall():
    print(row)

cursor.close()

('id', 'bigint', 'NO', 'PRI', None, 'auto_increment')
('company_id', 'int', 'NO', 'MUL', None, '')
('symbol', 'varchar(50)', 'NO', '', None, '')
('isin', 'varchar(20)', 'NO', '', None, '')
('company_name', 'varchar(255)', 'NO', '', None, '')
('headline', 'text', 'NO', '', None, '')
('feed_type', 'varchar(100)', 'NO', '', None, '')
('published_at', 'datetime', 'YES', '', None, '')
('event_date', 'date', 'YES', '', None, '')
('date_status', 'varchar(30)', 'NO', '', None, '')
('url', 'text', 'NO', '', None, '')
('source', 'varchar(50)', 'NO', '', None, '')
('created_at', 'timestamp', 'YES', '', 'CURRENT_TIMESTAMP', 'DEFAULT_GENERATED')


True

In [319]:
insert_query = """
INSERT INTO news (
    company_id,
    symbol,
    isin,
    company_name,
    headline,
    feed_type,
    published_at,
    event_date,
    date_status,
    url,
    source
)
VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
)
"""

cursor = db.cursor()

inserted = 0

for _, row in final_news.iterrows():

    values = (
        int(row["company_id"]),
        row["symbol"],
        row["isin"],
        row["company_name"],
        row["headline"],
        row["feed_type"],
        None if pd.isna(row["published_at"]) else row["published_at"].to_pydatetime(),
        None if pd.isna(row["event_date"]) else row["event_date"].date(),
        row["date_status"],
        row["url"],
        row["source"]
    )

    cursor.execute(insert_query, values)
    inserted += 1

db.commit()
cursor.close()

print("Inserted records:", inserted)

Inserted records: 66


In [320]:
query = """
SELECT
    COUNT(*) AS total_records
FROM news
"""

check_df = pd.read_sql(query, db)

check_df

,total_records
0,132


In [324]:
# Create a NEW cursor from the NEW connection

cursor = db.cursor()

print("DB connected:", db.is_connected())
print("Cursor created:", cursor is not None)

DB connected: True
Cursor created: True


In [325]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS news (
    id BIGINT AUTO_INCREMENT PRIMARY KEY,
    company_id INT NOT NULL,
    symbol VARCHAR(50) NOT NULL,
    isin VARCHAR(20) NOT NULL,
    company_name VARCHAR(255) NOT NULL,
    headline TEXT NOT NULL,
    feed_type VARCHAR(100) NOT NULL,
    published_at DATETIME NULL,
    event_date DATE NULL,
    date_status VARCHAR(50) NOT NULL,
    url TEXT NOT NULL,
    source VARCHAR(50) NOT NULL,

    INDEX idx_company_id (company_id),
    INDEX idx_feed_type (feed_type),
    INDEX idx_published_at (published_at),
    INDEX idx_event_date (event_date)
)
""")

db.commit()

print("news table created successfully")

news table created successfully


In [326]:
cursor.execute("SHOW TABLES")

for table in cursor.fetchall():
    print(table[0])

companies
industries
news
sectors
stock_prices


In [335]:
# Check data before inserting

print("Records to insert:", len(news_to_insert))
print(news_to_insert.columns.tolist())

Records to insert: 66
['company_id', 'symbol', 'isin', 'company_name', 'headline', 'feed_type', 'published_at', 'event_date', 'date_status', 'url', 'source']


In [342]:
insert_query = """
INSERT INTO news (
    company_id,
    symbol,
    isin,
    company_name,
    headline,
    feed_type,
    published_at,
    event_date,
    date_status,
    url,
    source
)
VALUES (
    %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s
)
"""

records = []

for _, row in news_to_insert.iterrows():

    records.append((
        int(row["company_id"]),
        row["symbol"],
        row["isin"],
        row["company_name"],
        row["headline"],
        row["feed_type"],
        row["published_at"].to_pydatetime()
            if pd.notna(row["published_at"]) else None,
        row["event_date"].date()
            if pd.notna(row["event_date"]) else None,
        row["date_status"],
        row["url"],
        row["source"]
    ))

cursor.executemany(insert_query, records)
db.commit()

print("Inserted records:", cursor.rowcount)

Inserted records: 66


In [343]:
cursor.execute("SELECT COUNT(*) FROM news")

count = cursor.fetchone()[0]

print("Total news records in DB:", count)

Total news records in DB: 66


In [344]:
cursor.execute("""
SELECT
    feed_type,
    COUNT(*) AS total
FROM news
GROUP BY feed_type
ORDER BY total DESC
""")

for row in cursor.fetchall():
    print(row)

('online_announcement', 28)
('annual_report', 20)
('related_party_transaction', 14)
('financial_result', 4)


In [345]:
cursor.execute("""
SELECT
    id,
    company_name,
    symbol,
    feed_type,
    published_at,
    event_date
FROM news
ORDER BY id DESC
LIMIT 10
""")

for row in cursor.fetchall():
    print(row)

(264, 'Supreme Facility Management Limited', 'SFML', 'related_party_transaction', datetime.datetime(2025, 2, 21, 15, 12, 33), None)
(263, 'Lloyds Luxuries Limited', 'LLOYDS', 'related_party_transaction', datetime.datetime(2025, 4, 25, 21, 12, 12), None)
(262, 'Ganesh Infraworld Limited', 'GANESHIN', 'related_party_transaction', datetime.datetime(2025, 4, 26, 19, 14, 11), None)
(261, 'Quest Laboratories Limited', 'QUESTLAB', 'related_party_transaction', datetime.datetime(2025, 4, 28, 14, 7, 7), None)
(260, 'Solex Energy Limited', 'SOLEX', 'related_party_transaction', datetime.datetime(2025, 5, 12, 22, 15, 35), None)
(259, 'Rex Pipes And Cables Industries Limited', 'REXPIPES', 'related_party_transaction', datetime.datetime(2025, 5, 15, 14, 41, 24), None)
(258, 'Integrated Personnel Services Limited', 'IPSL', 'related_party_transaction', datetime.datetime(2025, 5, 17, 0, 0, 33), None)
(257, 'Sp Refractories Limited', 'SPRL', 'related_party_transaction', datetime.datetime(2025, 5, 19, 16, 

In [346]:
cursor.execute("""
SELECT
    SUM(company_id IS NULL) AS missing_company_id,
    SUM(symbol IS NULL OR symbol = '') AS missing_symbol,
    SUM(headline IS NULL OR headline = '') AS missing_headline,
    SUM(feed_type IS NULL OR feed_type = '') AS missing_feed_type,
    SUM(url IS NULL OR url = '') AS missing_url
FROM news
""")

print(cursor.fetchone())

(Decimal('0'), Decimal('0'), Decimal('0'), Decimal('0'), Decimal('0'))


In [353]:
cursor.execute("DESCRIBE news")

for row in cursor.fetchall():
    print(row)

('id', 'bigint', 'NO', 'PRI', None, 'auto_increment')
('company_id', 'int', 'NO', 'MUL', None, '')
('symbol', 'varchar(50)', 'NO', '', None, '')
('isin', 'varchar(20)', 'NO', '', None, '')
('company_name', 'varchar(255)', 'NO', '', None, '')
('headline', 'text', 'NO', '', None, '')
('feed_type', 'varchar(100)', 'NO', '', None, '')
('published_at', 'datetime', 'YES', '', None, '')
('event_date', 'date', 'YES', '', None, '')
('date_status', 'varchar(30)', 'NO', '', None, '')
('url', 'text', 'NO', '', None, '')
('source', 'varchar(50)', 'NO', '', None, '')
('created_at', 'timestamp', 'YES', '', 'CURRENT_TIMESTAMP', 'DEFAULT_GENERATED')
